# Project Plan Aging - UBR Division - 2025

In [34]:
import pandas as pd
import numpy as np
import sqlite3

### 1. Loading Data from Database

In [35]:
conn = sqlite3.connect("data/machine_readings.db")
oee_daily = pd.read_sql_query('''SELECT 
    timestamp,
    MachineName,
    DivisionName,
    RunTime,
    DownTime,
    ProductionActual,
    ProductionRated,
    SpeedRated,
    SpeedActual,
    PlannedTime,
    Availability,
    Performance,
    Quality,
    OEE
FROM oee_daily_2025
WHERE DivisionName = 'UBR'
  AND timestamp BETWEEN '2025-01-01' AND '2025-06-30'
ORDER BY timestamp;''', conn) # PlanName skipped for now due to many missing values

downtime_info = pd.read_sql_query('''SELECT 
FromTime,
ToTime,
TotalTime,
MachineName,
DivisionName,
Reason
FROM downtime_info_2025
WHERE DivisionName = 'UBR'
AND TotalTime<=86400
AND FromTime BETWEEN '2025-01-01' AND '2025-06-30'
ORDER BY FromTime;''', conn)
conn.close()

### 2. Replacing "\N" to NaN

In [36]:
oee_daily = oee_daily.replace("\\N", np.nan).infer_objects(copy=False)
downtime_info = downtime_info.replace("\\N", np.nan).infer_objects(copy=False)

### 3. Observation

In [37]:
oee_daily.isnull().sum()

timestamp           0
MachineName         0
DivisionName        0
RunTime             0
DownTime            0
ProductionActual    0
ProductionRated     0
SpeedRated          0
SpeedActual         0
PlannedTime         0
Availability        0
Performance         0
Quality             0
OEE                 0
dtype: int64

In [38]:
downtime_info.isnull().sum()

FromTime            0
ToTime              0
TotalTime           0
MachineName         0
DivisionName        0
Reason          15002
dtype: int64

In [39]:
downtime_info.head()

,FromTime,ToTime,TotalTime,MachineName,DivisionName,Reason
0,2025-01-15 10:25:00.671404,2025-01-15 10:30:39.018265,338.346861,DT-1 U-4,UBR,NaN
1,2025-01-15 10:31:11.024194,2025-01-15 10:31:16.027208,5.003014,DT-1 U-4,UBR,Minor
2,2025-01-16 11:49:47.570889,2025-01-16 12:41:02.483996,3074.913107,DT-1 U-4,UBR,NaN
3,2025-01-16 12:50:00,2025-01-16 12:52:57.136564,177.136564,DT-1 U-4,UBR,Minor
4,2025-01-16 13:22:31.840799,2025-01-16 13:43:23.01768,1251.176881,DT-1 U-4,UBR,NaN


In [40]:
oee_daily.head()

,timestamp,MachineName,DivisionName,RunTime,DownTime,ProductionActual,ProductionRated,SpeedRated,SpeedActual,PlannedTime,Availability,Performance,Quality,OEE
0,2025-01-31 07:00:00,SioPlas U4,UBR,56825,5275,14280.0,0.0,0.0,16.44,62100,91.51,0.0,100.0,0.0
1,2025-01-31 07:00:00,48B Armouring,UBR,20837,41563,4888.0,0.0,0.0,5.03,62400,33.39,0.0,100.0,0.0
2,2025-01-31 07:00:00,LAYING UP 1+3,UBR,0,62400,2.0,0.0,0.0,0.00,62400,0.00,0.0,100.0,0.0
3,2025-01-31 07:00:00,Nokia 90 MM,UBR,4813,21887,1502.0,0.0,0.0,3.53,26700,18.03,0.0,100.0,0.0
4,2025-01-31 07:00:00,SioPlas U4,UBR,56825,5275,14280.0,0.0,0.0,16.44,62100,91.51,0.0,100.0,0.0


In [41]:
oee_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6709 entries, 0 to 6708
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         6709 non-null   object 
 1   MachineName       6709 non-null   object 
 2   DivisionName      6709 non-null   object 
 3   RunTime           6709 non-null   int64  
 4   DownTime          6709 non-null   int64  
 5   ProductionActual  6709 non-null   float64
 6   ProductionRated   6709 non-null   float64
 7   SpeedRated        6709 non-null   float64
 8   SpeedActual       6709 non-null   float64
 9   PlannedTime       6709 non-null   int64  
 10  Availability      6709 non-null   float64
 11  Performance       6709 non-null   float64
 12  Quality           6709 non-null   float64
 13  OEE               6709 non-null   float64
dtypes: float64(8), int64(3), object(3)
memory usage: 733.9+ KB


In [42]:
downtime_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157196 entries, 0 to 157195
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   FromTime      157196 non-null  object 
 1   ToTime        157196 non-null  object 
 2   TotalTime     157196 non-null  float64
 3   MachineName   157196 non-null  object 
 4   DivisionName  157196 non-null  object 
 5   Reason        142194 non-null  object 
dtypes: float64(1), object(5)
memory usage: 7.2+ MB


### 4. Basic Cleaning

In [43]:
# OEE Daily
oee_daily['timestamp'] = pd.to_datetime(oee_daily['timestamp'], format='mixed')
oee_daily['MachineName'] = oee_daily['MachineName'].str.strip()
oee_daily['DivisionName'] = oee_daily['DivisionName'].str.strip()

# Downtime Info
downtime_info['FromTime'] = pd.to_datetime(downtime_info['FromTime'], format='mixed')
downtime_info['ToTime'] = pd.to_datetime(downtime_info['ToTime'], format='mixed')
downtime_info['MachineName'] = downtime_info['MachineName'].str.strip()
downtime_info['DivisionName'] = downtime_info['DivisionName'].str.strip()
downtime_info['Reason'] = downtime_info['Reason'].replace('Lunch Time', 'Lunch')
downtime_info['Reason'] = downtime_info['Reason'].str.strip()

# handling numerical data
numeric_cols = [
    'RunTime','DownTime',
    'ProductionActual','ProductionRated',
    'SpeedActual','SpeedRated',
    'Availability','Performance','Quality','OEE'
]

for col in numeric_cols:
    oee_daily[col] = pd.to_numeric(oee_daily[col], errors='coerce')

### Date Alignment

In [44]:
oee_daily['Date'] = oee_daily['timestamp'].dt.date
downtime_info['Date'] = downtime_info['FromTime'].dt.date

### 5. Handling **Reason** Column null values in **Downtime Info**(Unfinished)

In [45]:
# for reason in downtime_info['Reason']:
#     fromtime = downtime_info.loc[reason, 'FromTime']
#     totaltime = downtime_info.loc[reason, 'TotalTime']
#     minute = fromtime.minute
#     hour = fromtime.hour
#     if pd.isnull(reason):
#         if minute < 5:
#             downtime_info.loc[reason, 'Reason'] = "Minor"
#         elif minute < 15:
#             downtime_info.loc[reason, 'Reason'] = "Process Setting"
#         elif (20 <= minute <= 40) and ((12 <= hour <= 14) or (21 <= hour <= 23)):
#             downtime_info.loc[reason, 'Reason'] = "Lunch"
#         elif 30 <= minute and ((6*60 + 40 <= minute <= 7*60 + 20) or (18*60 + 40 <= minute <= 19*60 + 20)):
#             downtime_info.loc[reason, 'Reason'] = "Other"
#         else:
#             downtime_info.loc[reason, 'Reason'] = "Other"
    

In [46]:
downtime_info = downtime_info.dropna()

### DAILY DOWNTIME AGGREGATION

In [47]:
down_daily = downtime_info.groupby(['MachineName','Date']).agg(
    Daily_Downtime=('TotalTime','sum'),
    Stop_Count=('TotalTime','size')
).reset_index()

# Shift downtime by 1 day to avoid leaking next day's degradation info
down_daily['Date'] = down_daily['Date'] + pd.Timedelta(days=1)

df = oee_daily.merge(
    down_daily,
    on=['MachineName','Date'],
    how='left'
)

df[['Daily_Downtime','Stop_Count']] = df[['Daily_Downtime','Stop_Count']].fillna(0)

### Feature Engineering

In [48]:
df['Speed_Ratio'] = np.where(
    df['SpeedRated'] > 0,
    df['SpeedActual'] / df['SpeedRated'],
    0
)

df['Production_Eff'] = np.where(
    df['ProductionRated'] > 0,
    df['ProductionActual'] / df['ProductionRated'],
    0
)

df['Downtime_Ratio'] = np.where(
    (df['RunTime'] + df['Daily_Downtime']) > 0,
    df['Daily_Downtime'] / (df['RunTime'] + df['Daily_Downtime']),
    0
)

# Sort data by machine and date
df = df.sort_values(['MachineName','Date'])

# Lagged features (previous day metrics)
lag_features = ['OEE', 'RunTime', 'DownTime', 'ProductionActual', 'ProductionRated', 'SpeedActual', 'SpeedRated', 'Downtime_Ratio', 'Production_Eff', 'Speed_Ratio']

for feat in lag_features:
    df[f'{feat}_lag1'] = df.groupby('MachineName')[feat].shift(1)

# Drop rows where lag features are missing
df = df.dropna(subset=[f'{feat}_lag1' for feat in lag_features])

### CREATE TARGET (NEXT DAY DEGRADATION)

In [49]:
# Define target using the original day metrics (OEE < 70 & Downtime_Ratio > 0.25)
df['Current_Degrading'] = np.where(
    (df['OEE'] < 70) & (df['Downtime_Ratio'] > 0.25),
    1, 0
)

# Target = next day's degrading status
df['Target'] = df.groupby('MachineName')['Current_Degrading'].shift(-1)

# Drop rows with NaN target (last day for each machine)
df = df.dropna(subset=['Target'])